# Batch Evaluation & Semantic Re-ranking
This notebook runs the end-to-end evaluation of the Visual Product Search Engine on Kaggle.
It computes Recall@K, NDCG@K, and mAP@K for the 3 ablation conditions:
- **A**: Vision-only CLIP ($alpha=1.0$)
- **B**: Frozen CLIP + Frozen BLIP-2
- **C**: Fine-tuned CLIP + Frozen BLIP-2


In [1]:
!pip install -q omegaconf timm fairscale iopath decord webdataset pycocotools pycocoevalcap
!pip install -q --no-dependencies salesforce-lavis
!pip install -q transformers==4.38.2 open_clip_torch pinecone pandas Pillow tqdm scikit-learn accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 86.1 MB/s eta 

In [2]:
import os
import torch
import open_clip
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pinecone import Pinecone
from lavis.models import load_model_and_preprocess
import numpy as np
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__

Device: cuda


In [3]:
# ==============================================================================
# CONFIGURATIONS
# ==============================================================================
PINECONE_API_KEY  = "pcsk_REDACTED"  # Replace with your key
INDEX_NAME        = "vr-clothing-gallery"

DATASET_ROOT      = "/kaggle/input/datasets/vinay1706/deepfashion-cropped" # Verify path
CLIP_CHECKPOINT   = f"{DATASET_ROOT}/clip_best.pt"
QUERY_CSV         = f"{DATASET_ROOT}/query.csv"
GALLERY_CSV       = f"{DATASET_ROOT}/gallery.csv"
CAPTIONS_CSV      = f"{DATASET_ROOT}/blip2_captions_gallery.csv"
IMAGE_ROOT        = DATASET_ROOT

# Top-K settings
TOP_K_RETRIEVAL = 15
K_VALUES = [5, 10, 15]

# Ablation Configs to Evaluate: (Config Name, Namespace, Use Finetuned CLIP, Use BLIP-2 Reranking)
CONFIGS = [
    ("Config A (Vision-Only)", "frozen-alpha-1.0", False, False),
    ("Config B (Frozen CLIP, alpha=0.7)", "frozen-alpha-0.7", False, True),
    ("Config B (Frozen CLIP, alpha=0.5)", "frozen-alpha-0.5", False, True),
    ("Config C (Finetuned CLIP, alpha=0.7)", "finetuned-alpha-0.7", True, True),
    ("Config C (Finetuned CLIP, alpha=0.5)", "finetuned-alpha-0.5", True, True)
]


In [4]:
# ==============================================================================
# LOAD DATA
# ==============================================================================
query_df = pd.read_csv(QUERY_CSV)
gallery_df = pd.read_csv(GALLERY_CSV)
captions_df = pd.read_csv(CAPTIONS_CSV)

# Create a mapping from image_name to caption for the gallery
caption_map = dict(zip(captions_df['image_name'], captions_df['blip2_caption']))
relevance_count = gallery_df.groupby('item_id').size().to_dict()

print(f"Loaded {len(query_df)} query images.")
print(f"Loaded {len(gallery_df)} gallery images.")


Loaded 14218 query images.
Loaded 12612 gallery images.


In [5]:
# ==============================================================================
# LOAD MODELS
# ==============================================================================
print("Loading LAVIS BLIP-2 ITM Model...")
blip_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="blip2_image_text_matching", 
    model_type="pretrain", 
    is_eval=True, 
    device=device
)

# Fix LAVIS missing padding in batched inference
class PatchedTokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, *args, **kwargs):
        kwargs["padding"] = True
        return self.tokenizer(*args, **kwargs)
    def __getattr__(self, name):
        return getattr(self.tokenizer, name)

blip_model.tokenizer = PatchedTokenizer(blip_model.tokenizer)

# CLIP loading helper
def load_clip(use_finetuned):
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
    if use_finetuned:
        ckpt = torch.load(CLIP_CHECKPOINT, map_location=device)
        # BUG FIX: The checkpoint uses the key 'model_state', not 'model_state_dict'
        state_dict = ckpt.get("model_state", ckpt)
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
    return model.to(device).eval(), preprocess

print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)


Loading LAVIS BLIP-2 ITM Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

100%|██████████| 1.89G/1.89G [00:08<00:00, 248MB/s]


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

100%|██████████| 712M/712M [00:02<00:00, 268MB/s]


Connecting to Pinecone...


In [6]:
def compute_itm_scores_batched(img_tensor, candidate_captions):
    """
    Computes true ITM probabilities for a batch of candidate captions against a single image.
    Accepts a preprocessed image tensor instead of a raw PIL image.
    Returns a list of probabilities (0.0 to 1.0).
    """
    # Duplicate image tensor to match the batch size of captions
    img_batch = img_tensor.repeat(len(candidate_captions), 1, 1, 1)
    
    # Preprocess all candidate text captions
    txt_batch = [txt_processors["eval"](c) for c in candidate_captions]
    
    # Pass through the ITM model in a single batch
    with torch.no_grad():
        itm_output = blip_model({"image": img_batch, "text_input": txt_batch}, match_head="itm")
        itm_scores = torch.nn.functional.softmax(itm_output, dim=1)[:, 1].cpu().tolist()
        
    return itm_scores

In [7]:
def calculate_metrics(ranked_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    metrics = {}
    for k in k_values:
        top_k_items = ranked_item_ids[:k]

        # Recall@K: fraction of all relevant gallery items retrieved in top-K
        total_relevant = relevance_count.get(ground_truth_id, 1)
        relevant_in_top_k = sum(1 for item in top_k_items if item == ground_truth_id)
        metrics[f'Recall@{k}'] = relevant_in_top_k / total_relevant

        # NDCG@K: accumulate gain for every relevant item, normalize by IDCG
        dcg = 0.0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                dcg += 1.0 / math.log2(i + 2)

        n_ideal = min(total_relevant, k)
        idcg = sum(1.0 / math.log2(i + 2) for i in range(n_ideal))
        metrics[f'NDCG@{k}'] = dcg / idcg if idcg > 0 else 0.0

        # mAP@K: precision at every relevant position, averaged over total relevant items
        ap = 0.0
        relevant_count_so_far = 0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                relevant_count_so_far += 1
                precision_at_i = relevant_count_so_far / (i + 1)
                ap += precision_at_i
        metrics[f'mAP@{k}'] = ap / total_relevant

    return metrics

In [8]:
# ==============================================================================
# MAIN EVALUATION LOOP
# ==============================================================================
NUM_QUERIES_TO_EVALUATE = None

if NUM_QUERIES_TO_EVALUATE:
    eval_df = query_df.drop_duplicates(subset='item_id').sample(n=NUM_QUERIES_TO_EVALUATE, random_state=104)
else:
    eval_df = query_df.drop_duplicates(subset='item_id')

results_summary = []

for config_name, namespace, use_finetuned, use_reranking in CONFIGS:
    print(f"\n{'='*60}\nEvaluating {config_name}\n{'='*60}")

    clip_model, clip_preprocess = load_clip(use_finetuned)

    config_metrics = {f'Recall@{k}': [] for k in K_VALUES}
    config_metrics.update({f'NDCG@{k}': [] for k in K_VALUES})
    config_metrics.update({f'mAP@{k}': [] for k in K_VALUES})

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        query_image_path = os.path.join(IMAGE_ROOT, row['image_name'])
        gt_item_id = row['item_id']

        # 1. Load and process query image
        try:
            raw_image = Image.open(query_image_path).convert('RGB')
            processed_image = clip_preprocess(raw_image).unsqueeze(0).to(device)
            img_tensor = vis_processors["eval"](raw_image).unsqueeze(0).to(device)
        except Exception as e:
            continue

        # 2. Get CLIP Visual Embedding
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                query_embedding = clip_model.encode_image(processed_image)
                query_embedding = torch.nn.functional.normalize(query_embedding, dim=-1).cpu().numpy().tolist()[0]

        # 3. Retrieve Top-K candidates from Pinecone
        retrieval_response = index.query(
            vector=query_embedding,
            top_k=TOP_K_RETRIEVAL,
            namespace=namespace,
            include_metadata=True
        )

        candidates = retrieval_response['matches']

        # 4. Semantic Re-ranking
        if use_reranking:
            cand_images = [cand['metadata']['image_name'] for cand in candidates]
            cand_captions = [caption_map.get(img, "") for img in cand_images]

            itm_scores = compute_itm_scores_batched(img_tensor, cand_captions)

            scored_candidates = list(zip(candidates, itm_scores))
            scored_candidates.sort(key=lambda x: x[1], reverse=True)
            candidates = [item[0] for item in scored_candidates]

        # 5. Extract Ranked Item IDs
        ranked_item_ids = [cand['metadata']['item_id'] for cand in candidates]

        # 6. Calculate Metrics
        q_metrics = calculate_metrics(ranked_item_ids, gt_item_id, K_VALUES)

        for k, v in q_metrics.items():
            config_metrics[k].append(v)

    # Aggregate and print results
    print(f"\nResults for {config_name}:")
    summary_row = {"Config": config_name}
    for metric, values in config_metrics.items():
        mean_val = np.mean(values)
        summary_row[metric] = mean_val
        print(f"{metric}: {mean_val:.4f}")

    results_summary.append(summary_row)

    del clip_model
    torch.cuda.empty_cache()

# Display Final Summary Table
results_df = pd.DataFrame(results_summary)
display(results_df)


Evaluating Config A (Vision-Only)


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  0%|          | 0/3985 [00:00<?, ?it/s]


Results for Config A (Vision-Only):
Recall@5: 0.2511
Recall@10: 0.3011
Recall@15: 0.3301
NDCG@5: 0.2686
NDCG@10: 0.2832
NDCG@15: 0.2927
mAP@5: 0.2021
mAP@10: 0.2129
mAP@15: 0.2169

Evaluating Config B (Frozen CLIP, alpha=0.7)


...

  0%|          | 0/3985 [00:00<?, ?it/s]


Results for Config C (Finetuned CLIP, alpha=0.7):
Recall@5: 0.4802
Recall@10: 0.5857
Recall@15: 0.6249
NDCG@5: 0.4461
NDCG@10: 0.4854
NDCG@15: 0.4999
mAP@5: 0.3560
mAP@10: 0.3838
mAP@15: 0.3916

Evaluating Config C (Finetuned CLIP, alpha=0.5)


  0%|          | 0/3985 [00:00<?, ?it/s]


Results for Config C (Finetuned CLIP, alpha=0.5):
Recall@5: 0.4077
Recall@10: 0.5003
Recall@15: 0.5320
NDCG@5: 0.3824
NDCG@10: 0.4165
NDCG@15: 0.4281
mAP@5: 0.3040
mAP@10: 0.3261
mAP@15: 0.3320


,Config,Recall@5,Recall@10,Recall@15,NDCG@5,NDCG@10,NDCG@15,mAP@5,mAP@10,mAP@15
0,Config A (Vision-Only),0.251145,0.301114,0.330116,0.268570,0.283235,0.292737,0.202074,0.212861,0.216856
1,"Config B (Frozen CLIP, alpha=0.7)",0.301456,0.353510,0.377936,0.295660,0.314269,0.323692,0.227555,0.239545,0.243621
2,"Config B (Frozen CLIP, alpha=0.5)",0.312085,0.372356,0.399174,0.299101,0.320816,0.330876,0.231235,0.244464,0.248672
3,"Config C (Finetuned CLIP, alpha=0.7)",0.480174,0.585748,0.624888,0.446085,0.485436,0.499923,0.355998,0.383762,0.391647
4,"Config C (Finetuned CLIP, alpha=0.5)",0.407674,0.500255,0.532028,0.382371,0.416491,0.428091,0.304046,0.326096,0.331984
